### Milestone2

In [20]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")

print("Dataset loaded successfully")
df.head()


Dataset loaded successfully


,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,NaN,NaN
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,NaN,NaN
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,NaN,NaN
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,NaN,NaN
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,NaN,NaN


In [21]:
df.columns

Index(['id', 'sender', 'subject', 'body', 'priority', 'triage_label',
       'ideal_intent', 'ideal_tone'],
      dtype='object')

In [22]:
df.shape

(200, 8)

### Ground truth function 

In [32]:
def draft_ground_truth(text):
    text = str(text).lower()

    # Ideal Intent
    if any(word in text for word in [
        "urgent", "deadline", "overdue", "alert",
        "payment", "invoice", "security", "submit", "reminder"
    ]):
        intent = "notify"
    elif any(word in text for word in [
        "newsletter", "promotion", "sale", "offer",
        "unsubscribe", "marketing"
    ]):
        intent = "ignore"
    else:
        intent = "respond"

    # Ideal Tone
    if any(word in text for word in [
        "urgent", "alert", "overdue", "asap",
        "immediately", "security", "deadline"
    ]):
        tone = "urgent"
    elif any(word in text for word in [
        "please", "kindly", "thank you", "thanks"
    ]):
        tone = "polite"
    else:
        tone = "neutral"

    return intent, tone

In [ ]:
## Apply the draft_ground_truth function to the 'body' column
# For each email body, the function returns:
#   (ideal_intent, ideal_tone)
df[["ideal_intent", "ideal_tone"]] = df["body"].apply(
    lambda x: pd.Series(draft_ground_truth(x))
)

df[["body", "ideal_intent", "ideal_tone"]].head()

,body,ideal_intent,ideal_tone
0,Reminder: The client meeting is scheduled at 1...,notify,neutral
1,Your invoice of INR 25515.09 is due on 2025-12...,notify,polite
2,Reminder: The client meeting is scheduled at 1...,notify,neutral
3,"Hello team, please find the attached weekly re...",respond,polite
4,"Hello team, please find the attached weekly re...",respond,polite


In [ ]:
# Save the updated DataFrame back to the CSV file
# This permanently stores the generated ground-truth labels
df.to_csv("../data/sample_emails_with_triage_200.csv", index=False)
print("Dataset updated with ground truth")

Dataset updated with ground truth


In [ ]:
#Email Assistant Logic (Rule-based Agent)
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(word in text for word in [
        "urgent", "deadline", "alert", "overdue",
        "invoice", "payment", "security", "reminder"
    ]):
        return "notify", "urgent"

    elif any(word in text for word in [
        "thank you", "thanks", "appreciate"
    ]):
        return "ignore", "polite"

    else:
        return "respond", "neutral"

In [36]:
#Generate Predictions
predictions = []

for _, row in df.iterrows():
    action, tone = email_assistant(row["body"])
    predictions.append({
        "id": row["id"],
        "predicted_intent": action,
        "predicted_tone": tone
    })

pred_df = pd.DataFrame(predictions)
pred_df.head()


,id,predicted_intent,predicted_tone
0,1,notify,urgent
1,2,notify,urgent
2,3,notify,urgent
3,4,respond,neutral
4,5,respond,neutral


In [38]:
#Merge Predictions with Ground Truth
eval_df = df.copy()
eval_df["predicted_intent"] = pred_df["predicted_intent"]
eval_df["predicted_tone"] = pred_df["predicted_tone"]

eval_df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone,predicted_intent,predicted_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,neutral,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,notify,polite,notify,urgent
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,neutral,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite,respond,neutral
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite,respond,neutral


In [39]:
#Evaluation function 
def evaluate(row):
    score = 0
    if row["predicted_intent"] == row["ideal_intent"]:
        score += 1
    if row["predicted_tone"] == row["ideal_tone"]:
        score += 1
    return score

In [40]:
#Compute accuracy 
eval_df["score"] = eval_df.apply(evaluate, axis=1)
accuracy = (eval_df["score"].sum() / (len(eval_df) * 2)) * 100
accuracy


np.float64(78.0)

In [41]:
#saving the output
eval_df.to_csv(
    "../data/milestone2_output_theertha.csv",
    index=False
)
